# Telugu FastSpeech 2 (IndicTTS) — Kaggle

IIIT-H Speech Processing Lab. Deadline Saturday 7 November 2026, midnight IST.

Train the **Telugu** model only. It does not load the English checkpoint.

**Before Run all:**

1. **Accelerator → GPU T4**, **Internet → On**.
2. IndicTTS Telugu is public, so `HF_TOKEN` is optional. The English notebook needs it for Svarah. If you already added the Kaggle secret, this notebook will load it and will not print it.
3. IndicTTS license: https://www.iitm.ac.in/donlab/indictts/downloads/license.pdf

The hub card is two speakers (female ~4.27 h, male ~4.47 h). Female clips in the preview are often 16 kHz paragraphs longer than 12 s; male clips are often 48 kHz and shorter. The download splits long clips into ≤12 s slices. Read `data/telugu/split_stats.json` for the hours actually written. If `reached_target` is false, put those hours in the report. Do not assume 5 h / 1 h.


## What this run does

Download and subset IndicTTS Telugu, align syllables with MMS (`facebook/mms-1b-fl102`, language `tel`), extract features, train up to 20,000 steps into `Telugu/checkpoints`, then write `Results/Telugu/` (loss curves, T1–T4 wavs, duration / pitch / energy plots).

Resume works the same way as the English notebook: rerun, or set `FS2_EXTRA_CKPT` to a previous output.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/flyme2mars/iiith-fastspeech2-submission.git"
BRANCH = "main"
on_kaggle = Path("/kaggle/working").exists()
dest = Path("/kaggle/working/iiith-fastspeech2-assignment") if on_kaggle else Path.cwd()

if not on_kaggle and not (dest / "English" / "train.py").exists() and (Path.cwd() / "English" / "train.py").exists():
    dest = Path.cwd()

if on_kaggle:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        !git clone {REPO_URL} {dest}
    %cd {dest}
    !git fetch origin {BRANCH}
    !git checkout -B {BRANCH} origin/{BRANCH}
    !git reset --hard origin/{BRANCH}
else:
    %cd {dest}
    print("Not on Kaggle; leaving the git checkout alone. Use branch", BRANCH)

print("cwd", Path.cwd())
import torch
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
elif not os.environ.get("FS2_ALLOW_CPU"):
    raise SystemExit("Turn on Accelerator -> GPU T4 before running the rest of this notebook.")


In [ ]:
# Leave Kaggle's CUDA torch/torchaudio installed. requirements.txt would replace them.
!pip install -q -r requirements-kaggle.txt


In [ ]:
import os
os.environ.pop("HF_TOKEN", None)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets. The value is not printed.")
except Exception as exc:
    print("HF_TOKEN was not loaded:", type(exc).__name__)
    print("Svarah is gated. Add-ons -> Secrets -> label HF_TOKEN.")
    print("Accept the dataset first: https://huggingface.co/datasets/ai4bharat/Svarah")
    print("Telugu (IndicTTS) does not need this secret.")


## Download IndicTTS Telugu

`HF_TOKEN` is not required. `--force` rewrites the wavs and the manifest from the cached duration index, which is what a rerun after the alignment fix needs. Add `--reindex` only if that index is missing or wrong. The script prints the real train and val hours. A previous run that died mid-align did **not** measure the corpus; it extracted about 3.19 h and then found an empty validation split.


In [ ]:
!python scripts/download_indictts.py --force


## Align and extract features

CTC with the Telugu MMS adapter. A clip whose transcript is longer than the acoustic frames is skipped and listed in `preprocessed_data/telugu/align_report.json`. One failure does not stop the run. `--aligner mfa` is rejected: there is no Telugu MFA acoustic model in this repo. The MMS download is large (on the order of a few GB).

New slice names look like `te_00021_00`. TextGrids from the failed run (`te_00021` with no suffix) are not reused. If disk is tight, delete `preprocessed_data/telugu/TextGrid` before this cell.


In [ ]:
!python scripts/align.py --lang telugu --aligner ctc && python scripts/extract_features.py --lang telugu


In [ ]:
!python scripts/download_vocoder.py


## Train

Same step budget as English (20,000, checkpoint every 2,000). The speaker table is `female` / `male`. If the T4 OOMs, set batch size 8 and `grad_acc_step` 2 in `configs/telugu.yaml` and resume.


In [ ]:
import os
# os.environ["FS2_EXTRA_CKPT"] = "/kaggle/input/<previous-output>/Telugu/checkpoints"
!python scripts/train.py --lang telugu


In [ ]:
!python scripts/plot_losses.py --lang telugu
!python scripts/synthesize.py --lang telugu
!python scripts/plot_variance.py --lang telugu


## Files to bring back

- `Telugu/checkpoints/*.pth.tar`
- `Results/Telugu/training_loss.png`, `validation_loss.png`
- `Results/Telugu/test_generated_audios/{T1,T2a,T2b,T3,T4}.wav`
- `Results/Telugu/variance_adaptor_plots/{duration,pitch,energy}/*.png`
- `data/telugu/split_stats.json`

Keep these separate from the English results. Fill the Telugu rows in `report/REPORT.md` from the logs, not from memory.


In [ ]:
from pathlib import Path
root = Path("Results/Telugu")
for path in sorted(root.rglob("*")):
    if path.is_file() and path.suffix != ".npz":
        print(path)
